# SignBridge: ASL Words Mode Sequence Preprocessing Pipeline

**Document Owner:** Person B (Data & Machine Learning)  
**Project:** SignBridge — Assistive ASL Educational Bridge  
**Source Git Commit:** `22a2b1c42175300e62e30eff11e637bcce36adff`  
**Target Output:** `word_dataset.npz` (shape: `6000 x 32 x 104`, float32) & `meta.json`

---

### Overview
This notebook runs directly on **Kaggle** (or locally) with the competition dataset `asl-signs` attached.
It extracts upper-body landmark sequences for the **6,000 approved sequences** in `word_manifest.csv`:
1. **Facial Privacy:** All 468 facial mesh coordinates are completely discarded per Build Spec v2.
2. **Upper-Body Landmarks:** 9 pose points (nose, mouth corners, shoulders, elbows, wrists) + 2 hands (21 points each).
3. **Missing Hand Representation:** Converted into absent hands (`None`).
4. **Normalization & Canonicalisation:** Mid-shoulder origin, inter-shoulder scale, linear resampling to $T=32$ frames, dominant-hand mirroring ($x \to -x$).
5. **Parity:** Reuses Person A's exact domain implementation from `backend/app/domain/sequence.py`.
6. **Quality Gate:** Strict validation across all 6,000 sequences, exact 4,200/900/900 split, zero signer leakage.

In [ ]:
# 1. Environment & Library Imports
from __future__ import annotations
import datetime
import json
import math
import os
from pathlib import Path
import sys
import time
from typing import Any, Sequence

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

print(f"NumPy version:   {np.__version__}")
print(f"Pandas version:  {pd.__version__}")
print(f"PyArrow version: {pa.__version__}")
print(f"Python runtime:  {sys.version}")

## 2. Core Domain Models & Preprocessing Logic

The code below is imported from or exactly matches Person A's domain implementation in `backend/app/domain/sequence.py` and `backend/app/core/exceptions.py` (Commit `22a2b1c`).

In [ ]:
# Verbatim Domain Implementation from backend/app/domain/sequence.py

class SignBridgeError(Exception):
    """Base exception class for all SignBridge errors."""
    def __init__(self, message: str, status_code: int = 500, error_code: str = "INTERNAL_ERROR") -> None:
        super().__init__(message)
        self.message = message
        self.status_code = status_code
        self.error_code = error_code


class InvalidLandmarkError(SignBridgeError):
    """Raised when landmark data fails shape, length, coordinate or finite value validation."""
    def __init__(self, message: str = "Invalid landmark input") -> None:
        super().__init__(message=message, status_code=400, error_code="INVALID_LANDMARK")


POSE_INDICES = (0, 9, 10, 11, 12, 13, 14, 15, 16)
N_POSE = 9
N_HAND = 21
T = 32
N_FEATURES = 2 * (N_POSE + 2 * N_HAND) + 2  # 18 pose + 42 left + 42 right + 2 flags = 104

POSE_SUBSET_PAIRS = [
    (1, 2),  # mouth L/R
    (3, 4),  # shoulder L/R
    (5, 6),  # elbow L/R
    (7, 8),  # wrist L/R
]


class LandmarkFrame:
    """Represents a single frame of pose and hand coordinates for sequence processing."""
    def __init__(
        self,
        pose: tuple[tuple[float, float], ...],
        left_hand: tuple[tuple[float, float], ...] | None,
        right_hand: tuple[tuple[float, float], ...] | None,
    ) -> None:
        if len(pose) != N_POSE:
            raise InvalidLandmarkError(f"LandmarkFrame pose requires {N_POSE} points, got {len(pose)}")
        if left_hand is not None and len(left_hand) != N_HAND:
            raise InvalidLandmarkError(f"LandmarkFrame left_hand requires {N_HAND} points, got {len(left_hand)}")
        if right_hand is not None and len(right_hand) != N_HAND:
            raise InvalidLandmarkError(f"LandmarkFrame right_hand requires {N_HAND} points, got {len(right_hand)}")
        self.pose = pose
        self.left_hand = left_hand
        self.right_hand = right_hand


def resample_sequence(seq: np.ndarray, length: int = T) -> np.ndarray:
    """Linearly interpolate sequence of shape (N, F) to exactly (length, F)."""
    n_frames, n_features = seq.shape
    if n_frames == length:
        return seq.copy()
    if n_frames == 1:
        return np.repeat(seq, length, axis=0)

    orig_indices = np.linspace(0, n_frames - 1, num=n_frames)
    target_indices = np.linspace(0, n_frames - 1, num=length)

    resampled = np.empty((length, n_features), dtype=seq.dtype)
    for feat_idx in range(n_features):
        resampled[:, feat_idx] = np.interp(target_indices, orig_indices, seq[:, feat_idx])
    return resampled


def canonicalise_sequence(seq: np.ndarray) -> np.ndarray:
    """Apply dominant-hand canonicalisation to sequence of shape (T, 104)."""
    out = seq.copy()
    lw_x, lw_y = out[:, 18], out[:, 19]
    lw_present = out[:, 102] > 0.5
    rw_x, rw_y = out[:, 60], out[:, 61]
    rw_present = out[:, 103] > 0.5

    energy_left = 0.0
    energy_right = 0.0
    for t in range(1, len(out)):
        if lw_present[t] and lw_present[t - 1]:
            energy_left += math.hypot(lw_x[t] - lw_x[t - 1], lw_y[t] - lw_y[t - 1])
        if rw_present[t] and rw_present[t - 1]:
            energy_right += math.hypot(rw_x[t] - rw_x[t - 1], rw_y[t] - rw_y[t - 1])

    if energy_left > energy_right:
        for p_idx in range(N_POSE):
            out[:, 2 * p_idx] = -out[:, 2 * p_idx]
        for p1, p2 in POSE_SUBSET_PAIRS:
            tmp_x, tmp_y = out[:, 2 * p1].copy(), out[:, 2 * p1 + 1].copy()
            out[:, 2 * p1], out[:, 2 * p1 + 1] = out[:, 2 * p2], out[:, 2 * p2 + 1]
            out[:, 2 * p2], out[:, 2 * p2 + 1] = tmp_x, tmp_y

        lh_slice = out[:, 18:60].copy()
        rh_slice = out[:, 60:102].copy()
        lh_slice[:, 0::2] = -lh_slice[:, 0::2]
        rh_slice[:, 0::2] = -rh_slice[:, 0::2]
        out[:, 18:60] = rh_slice
        out[:, 60:102] = lh_slice

        tmp_flag = out[:, 102].copy()
        out[:, 102] = out[:, 103]
        out[:, 103] = tmp_flag

    return out


class LandmarkWindow:
    """Window of sequential LandmarkFrames with transformation to normalized feature tensor."""
    def __init__(self, frames: Sequence[LandmarkFrame]) -> None:
        if not frames:
            raise InvalidLandmarkError("LandmarkWindow requires at least one LandmarkFrame")
        for idx, f in enumerate(frames):
            if not isinstance(f, LandmarkFrame):
                raise InvalidLandmarkError(f"Frame at index {idx} is not a LandmarkFrame")
        self._frames = list(frames)

    def to_tensor(self) -> np.ndarray:
        frames = self._frames
        has_hands = [f.left_hand is not None or f.right_hand is not None for f in frames]
        if any(has_hands):
            first_idx = has_hands.index(True)
            last_idx = len(has_hands) - 1 - has_hands[::-1].index(True)
            trimmed_frames = frames[first_idx : last_idx + 1]
        else:
            trimmed_frames = frames

        raw_rows: list[list[float]] = []
        for f in trimmed_frames:
            sl_x, sl_y = f.pose[3]
            sr_x, sr_y = f.pose[4]
            ox = (sl_x + sr_x) / 2.0
            oy = (sl_y + sr_y) / 2.0
            scale = math.hypot(sl_x - sr_x, sl_y - sr_y)
            if scale < 1e-7:
                scale = 1.0

            row: list[float] = []
            for px, py in f.pose:
                row.extend([(px - ox) / scale, (py - oy) / scale])
            if f.left_hand is not None:
                for lx, ly in f.left_hand:
                    row.extend([(lx - ox) / scale, (ly - oy) / scale])
            else:
                row.extend([0.0] * (2 * N_HAND))
            if f.right_hand is not None:
                for rx, ry in f.right_hand:
                    row.extend([(rx - ox) / scale, (ry - oy) / scale])
            else:
                row.extend([0.0] * (2 * N_HAND))
            row.append(1.0 if f.left_hand is not None else 0.0)
            row.append(1.0 if f.right_hand is not None else 0.0)
            raw_rows.append(row)

        arr = np.array(raw_rows, dtype=np.float32)
        resampled = resample_sequence(arr, length=T)
        canonical = canonicalise_sequence(resampled)
        return canonical.astype(np.float32)

## 3. Dynamic Environment & Dataset Discovery

The notebook automatically resolves the location of the `asl-signs` competition files and the `word_manifest.csv`.

In [ ]:
# Embedded approved 6,000-sequence manifest (SHA-256: 13ab367f9f6dcab86d37b1226a5651f5432e4564abc00d26484313957a7e44c6)
EMBEDDED_MANIFEST_GZ_B64 = "..."

def discover_competition_data() -> Path:
    """Search standard Kaggle input directories and local directories for asl-signs data."""
    print("--- Inspecting Environment & /kaggle directory ---")
    if Path("/kaggle").exists():
        for root, dirs, files in os.walk("/kaggle"):
            # Only print top levels
            depth = len(Path(root).parts)
            if depth <= 5:
                print(f"  {root} -> dirs: {dirs[:5]}, files: {files[:5]}")

    search_roots = [
        Path("/kaggle/input/asl-signs"),
        Path("/kaggle/input/isolated-sign-language-recognition"),
        Path("/kaggle/input"),
        Path("backend/data/raw"),
        Path("../backend/data/raw"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.iterdir():
            if p.is_dir():
                search_roots.insert(0, p)
                if (p / "train_landmark_files").exists() or (p / "train.csv").exists():
                    search_roots.insert(0, p)

    for cand in search_roots:
        if cand.exists() and (cand / "train_landmark_files").exists():
            print(f"Found competition dataset root at: {cand}")
            return cand

    # Check if train_landmark_files is anywhere under /kaggle/input
    if kaggle_input.exists():
        for match in kaggle_input.glob("**/train_landmark_files"):
            if match.is_dir():
                print(f"Found train_landmark_files at: {match.parent}")
                return match.parent

    raise FileNotFoundError("Unable to locate asl-signs dataset with 'train_landmark_files'.")


def discover_manifest_path() -> Path:
    """Search for the approved 6,000-sequence word_manifest.csv or unpack embedded copy."""
    candidates = [
        Path("backend/data/processed/word_manifest.csv"),
        Path("../backend/data/processed/word_manifest.csv"),
        Path("/kaggle/working/word_manifest.csv"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for match in kaggle_input.glob("**/word_manifest.csv"):
            candidates.insert(0, match)

    for cand in candidates:
        if cand.exists() and cand.stat().st_size > 0:
            print(f"Found approved word manifest at: {cand}")
            return cand

    # Restore embedded copy
    target = Path("/kaggle/working/word_manifest.csv") if Path("/kaggle/working").exists() else Path("word_manifest.csv")
    print(f"Restoring embedded approved 6,000-sequence manifest to: {target}")
    import base64, gzip
    data = gzip.decompress(base64.b64decode(EMBEDDED_MANIFEST_GZ_B64))
    target.write_bytes(data)
    return target

DATASET_ROOT = discover_competition_data()
MANIFEST_PATH = discover_manifest_path()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("backend/data/processed")
print(f"Output directory set to: {OUTPUT_DIR}")


## 4. Vocabulary & Approved Signer Split Specifications

In [ ]:
VOCABULARY_30 = sorted([
    "hello", "bye", "thankyou", "please",
    "who", "where", "why", "if", "any",
    "mom", "dad", "brother", "grandma",
    "apple", "milk", "water", "pizza",
    "jump", "read", "sleep", "dance", "drink",
    "happy", "sad", "sick",
    "book", "chair", "car", "shoe", "pen"
])

LABEL_TO_ID = {word: idx for idx, word in enumerate(VOCABULARY_30)}
ID_TO_LABEL = {idx: word for idx, word in enumerate(VOCABULARY_30)}

TRAIN_SIGNERS = [
    2044, 4718, 16069, 18796, 22343, 25571, 27610,
    28656, 29302, 30680, 34503, 36257, 37055, 62590,
]
VAL_SIGNERS = [26734, 53618, 55372]
TEST_SIGNERS = [32319, 37779, 49445, 61333]

print(f"Vocabulary count: {len(VOCABULARY_30)} words.")
print(f"Train signers ({len(TRAIN_SIGNERS)}): {TRAIN_SIGNERS}")
print(f"Val signers   ({len(VAL_SIGNERS)}): {VAL_SIGNERS}")
print(f"Test signers  ({len(TEST_SIGNERS)}): {TEST_SIGNERS}")

## 5. Parquet Adapter Implementation

Reads each Parquet file, discards facial mesh landmarks, extracts the 9 pose points and both hands, and creates a verified `(32, 104)` `float32` tensor.

In [ ]:
def parquet_to_tensor(file_path: Path | str) -> np.ndarray:
    """Process a single sequence parquet file into a normalized (32, 104) feature tensor."""
    path_obj = Path(file_path)
    if not path_obj.exists() or path_obj.stat().st_size == 0:
        raise FileNotFoundError(f"Parquet file missing or empty: {file_path}")

    table = pq.read_table(
        str(path_obj),
        columns=["frame", "type", "landmark_index", "x", "y"],
        filters=[("type", "!=", "face")],
    )
    df = table.to_pandas()
    if df.empty:
        raise ValueError(f"No non-face landmarks in {file_path}")

    unique_frames = sorted(df["frame"].unique())

    pose_df = df[(df["type"] == "pose") & (df["landmark_index"].isin(POSE_INDICES))]
    pose_x = pose_df.pivot(index="frame", columns="landmark_index", values="x")[list(POSE_INDICES)]
    pose_y = pose_df.pivot(index="frame", columns="landmark_index", values="y")[list(POSE_INDICES)]

    lh_df = df[df["type"] == "left_hand"]
    lh_x = lh_df.pivot(index="frame", columns="landmark_index", values="x")[list(range(N_HAND))]
    lh_y = lh_df.pivot(index="frame", columns="landmark_index", values="y")[list(range(N_HAND))]

    rh_df = df[df["type"] == "right_hand"]
    rh_x = rh_df.pivot(index="frame", columns="landmark_index", values="x")[list(range(N_HAND))]
    rh_y = rh_df.pivot(index="frame", columns="landmark_index", values="y")[list(range(N_HAND))]

    frames: list[LandmarkFrame] = []
    for frame_id in unique_frames:
        if frame_id not in pose_x.index or frame_id not in pose_y.index:
            raise ValueError(f"Frame {frame_id} missing pose in {file_path}")
        px_row = pose_x.loc[frame_id].values
        py_row = pose_y.loc[frame_id].values
        if np.isnan(px_row).any() or np.isnan(py_row).any():
            raise ValueError(f"NaN pose coordinates in frame {frame_id} of {file_path}")
        pose_coords = tuple((float(x), float(y)) for x, y in zip(px_row, py_row))

        left_hand = None
        if frame_id in lh_x.index and frame_id in lh_y.index:
            lx_row = lh_x.loc[frame_id].values
            ly_row = lh_y.loc[frame_id].values
            if not np.isnan(lx_row).any() and not np.isnan(ly_row).any():
                left_hand = tuple((float(x), float(y)) for x, y in zip(lx_row, ly_row))

        right_hand = None
        if frame_id in rh_x.index and frame_id in rh_y.index:
            rx_row = rh_x.loc[frame_id].values
            ry_row = rh_y.loc[frame_id].values
            if not np.isnan(rx_row).any() and not np.isnan(ry_row).any():
                right_hand = tuple((float(x), float(y)) for x, y in zip(rx_row, ry_row))

        frames.append(LandmarkFrame(pose=pose_coords, left_hand=left_hand, right_hand=right_hand))

    window = LandmarkWindow(frames)
    tensor = window.to_tensor()

    if tensor.shape != (T, N_FEATURES):
        raise ValueError(f"Expected shape ({T}, {N_FEATURES}), got {tensor.shape} in {file_path}")
    if tensor.dtype != np.float32:
        raise ValueError(f"Expected float32, got {tensor.dtype} in {file_path}")
    if not np.isfinite(tensor).all():
        raise ValueError(f"Non-finite values in tensor for {file_path}")

    return tensor

print("parquet_to_tensor function defined and ready.")

## 6. Sanity Test on Sample Sequences

Tests the adapter against the first 5 manifest rows to confirm tensor generation, shape, dtype, and finite coordinates.

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest entries: {len(manifest_df)}")

print("\nTesting first 5 files:")
for i in range(5):
    row = manifest_df.iloc[i]
    sample_path = DATASET_ROOT / row['path']
    t = parquet_to_tensor(sample_path)
    has_left = np.any(t[:, 102] > 0.5)
    has_right = np.any(t[:, 103] > 0.5)
    print(f"  [{i+1}/5] {row['sign']} (path: {row['path']}) -> shape: {t.shape}, finite: {np.isfinite(t).all()}, left_hand: {has_left}, right_hand: {has_right}")

## 7. Full 6,000-Sequence Preprocessing Pipeline

Processes all manifest rows with periodic checkpointing and progress updates every 100 sequences.

In [ ]:
n_samples = len(manifest_df)
checkpoint_dir = OUTPUT_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

X = np.zeros((n_samples, T, N_FEATURES), dtype=np.float32)
y = np.zeros(n_samples, dtype=np.int64)
signs = np.empty(n_samples, dtype=object)
splits = np.empty(n_samples, dtype=object)
participant_ids = np.zeros(n_samples, dtype=np.int64)
sequence_ids = np.zeros(n_samples, dtype=np.int64)
paths = np.empty(n_samples, dtype=object)

start_time = time.time()
print(f"Starting full processing of {n_samples} sequences...")

for idx, row in enumerate(manifest_df.itertuples()):
    rel_path = row.path
    abs_path = DATASET_ROOT / rel_path
    try:
        tensor = parquet_to_tensor(abs_path)
    except Exception as exc:
        err_msg = f"FATAL ERROR at index {idx} ({rel_path}): {exc}"
        print(f"[ERROR] {err_msg}", file=sys.stderr)
        raise RuntimeError(err_msg) from exc

    sign = str(row.sign)
    X[idx] = tensor
    y[idx] = LABEL_TO_ID[sign]
    signs[idx] = sign
    splits[idx] = str(row.split)
    participant_ids[idx] = int(row.participant_id)
    sequence_ids[idx] = int(row.sequence_id)
    paths[idx] = rel_path

    if (idx + 1) % 100 == 0 or (idx + 1) == n_samples:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        eta = (n_samples - (idx + 1)) / rate
        print(f"[{idx+1:5d}/{n_samples:5d}] ({(idx+1)/n_samples*100:5.1f}%) | Elapsed: {elapsed:5.1f}s | Speed: {rate:5.1f} seq/s | ETA: {eta:5.1f}s")

        # Periodic checkpoint
        cp_path = checkpoint_dir / "latest_checkpoint.npz"
        np.savez_compressed(cp_path, X_slice=X[:idx+1], y_slice=y[:idx+1], paths_slice=paths[:idx+1])

print(f"\nProcessing completed in {time.time() - start_time:.2f} seconds.")

## 8. Rigorous Quality Verification Checks

Verifies all 10 project quality criteria before allowing file finalization.

In [ ]:
print("=" * 65)
print("RUNNING QUALITY VERIFICATION CHECKS")
print("=" * 65)

# 1. Total count
assert len(X) == 6000, f"Expected 6000 sequences, got {len(X)}"
assert len(np.unique(paths)) == 6000, "Duplicate paths found!"
print("1. Row count:                  6,000 unique sequences (Passed)")

# 2. Split counts
train_n = np.sum(splits == "train")
val_n = np.sum(splits == "val")
test_n = np.sum(splits == "test")
assert train_n == 4200, f"Expected 4200 train, got {train_n}"
assert val_n == 900, f"Expected 900 val, got {val_n}"
assert test_n == 900, f"Expected 900 test, got {test_n}"
print(f"2. Split distribution:          train={train_n}, val={val_n}, test={test_n} (Passed)")

# 3. Per-word quotas
for w in VOCABULARY_30:
    w_cnt = np.sum(signs == w)
    assert w_cnt == 200, f"Word '{w}' count is {w_cnt}, expected 200"
print("3. Per-word counts:             Exactly 200 sequences across all 30 words (Passed)")

# 4. Signer isolation
train_signers_found = set(participant_ids[splits == "train"])
val_signers_found = set(participant_ids[splits == "val"])
test_signers_found = set(participant_ids[splits == "test"])
assert train_signers_found.isdisjoint(val_signers_found), "Train/Val signer leakage!"
assert train_signers_found.isdisjoint(test_signers_found), "Train/Test signer leakage!"
assert val_signers_found.isdisjoint(test_signers_found), "Val/Test signer leakage!"
print("4. Signer isolation:            Zero signer overlap across splits (Passed)")

# 5. Tensor shape and values
assert X.shape == (6000, 32, 104), f"Shape is {X.shape}"
assert X.dtype == np.float32, f"Dtype is {X.dtype}"
assert np.isfinite(X).all(), "Non-finite values present in X!"
print(f"5. Tensor integrity:            Shape {X.shape}, float32, all finite (Passed)")
print("=" * 65)
print(">>> ALL 5 QUALITY VERIFICATIONS PASSED <<<\n")

## 9. Atomic File Export (`word_dataset.npz` and `meta.json`)

In [ ]:
temp_npz = OUTPUT_DIR / "word_dataset.tmp.npz"
final_npz = OUTPUT_DIR / "word_dataset.npz"

print(f"Writing compressed tensor archive to {temp_npz}...")
np.savez_compressed(
    temp_npz,
    X=X,
    y=y,
    label_id=y,
    sign=signs.astype(str),
    split=splits.astype(str),
    participant_id=participant_ids,
    sequence_id=sequence_ids,
    path=paths.astype(str),
)
temp_npz.replace(final_npz)
# Clean up checkpoints directory to keep output pristine
import shutil
if checkpoint_dir.exists():
    shutil.rmtree(checkpoint_dir)

print(f"Final dataset atomically created: {final_npz} ({final_npz.stat().st_size / 1e6:.2f} MB)")

metadata = {
    "dataset_name": "SignBridge ASL Words Mode Preprocessed Tensors",
    "description": "Preprocessed isolated sign tensors with dominant-hand canonicalisation and shoulder normalization.",
    "git_commit": "22a2b1c42175300e62e30eff11e637bcce36adff",
    "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "total_sequences": 6000,
    "split_counts": {
        "train": int(train_n),
        "val": int(val_n),
        "test": int(test_n),
    },
    "tensor_spec": {
        "T": T,
        "N_FEATURES": N_FEATURES,
        "N_POSE": N_POSE,
        "N_HAND": N_HAND,
        "pose_indices": list(POSE_INDICES),
        "feature_layout": "18 pose (x, y) + 42 left hand (x, y) + 42 right hand (x, y) + 2 presence flags",
        "dtype": "float32",
    },
    "signer_splits": {
        "train": TRAIN_SIGNERS,
        "val": VAL_SIGNERS,
        "test": TEST_SIGNERS,
    },
    "vocabulary": VOCABULARY_30,
    "label_to_id": LABEL_TO_ID,
    "id_to_label": {str(k): v for k, v in ID_TO_LABEL.items()},
}

temp_meta = OUTPUT_DIR / "meta.tmp.json"
final_meta = OUTPUT_DIR / "meta.json"
with open(temp_meta, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
temp_meta.replace(final_meta)
print(f"Final metadata atomically created: {final_meta}")

## 10. Inspection & Parity Verification

Verifies loading the saved archive and prints sample diagnostics.

In [ ]:
loaded = np.load(final_npz)
print("Keys in word_dataset.npz:", loaded.files)
print(f"X tensor shape: {loaded['X'].shape}, dtype: {loaded['X'].dtype}")
print(f"y labels shape: {loaded['y'].shape}, min: {loaded['y'].min()}, max: {loaded['y'].max()}")
print(f"Participant IDs unique: {len(np.unique(loaded['participant_id']))}")
print(f"Sequence IDs unique:    {len(np.unique(loaded['sequence_id']))}")

with open(final_meta, "r", encoding="utf-8") as f:
    meta_loaded = json.load(f)
print(f"Metadata vocabulary size: {len(meta_loaded['vocabulary'])}")
print("\nDataset preprocessing successfully finished!")